[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-06-transformations.ipynb)

---
# Day 6 · Transformations and dbt Integration
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Transformations

> **Goal for today:** Master the ELT pattern with dlt. Run SQL transformations using dlt's built-in SQL client, apply in-flight record transforms, and wire dlt into a dbt project so your warehouse models stay in sync automatically.

---
## The ELT pattern: Extract → Load → Transform

Traditional ETL transforms data *before* loading it. Modern ELT loads raw data first, then transforms inside the warehouse. dlt is built for ELT:

| Stage | Who owns it | What happens |
|---|---|---|
| **Extract** | dlt `@dlt.resource` | Pull data from APIs, DBs, files |
| **Load** | dlt `pipeline.run()` | Normalise, type-coerce, write to destination |
| **Transform** | dlt SQL client **or dbt** | Build aggregates, clean, model the data |

**Tip:** dlt handles extraction and loading. dbt handles transformation. Together they form a clean ELT pattern. You get the best of both worlds — dlt's connector ecosystem and dbt's transformation lineage.

### Two transformation approaches

| Approach | When to use | How |
|---|---|---|
| **In-flight transform** | Clean/reshape records *before* they land | `@dlt.transformer` decorator |
| **Post-load SQL** | Aggregate and model *after* raw data is loaded | `pipeline.sql_client()` |
| **dbt** | Full transformation layer with lineage, tests, docs | Run `dbt run` after `pipeline.run()` |

---
## Step 1 · Install dlt

In [ ]:
%pip install -q "dlt[duckdb]"

---
## Step 2 · Generate a synthetic e-commerce dataset and load it

We'll create synthetic order data that mimics a real e-commerce platform. This is our raw layer — we load it as-is, then transform.

In [ ]:
import dlt
import random
from datetime import date, timedelta

# Synthetic data generators — these simulate pulling from a real order system
PRODUCTS = [
    {"product_id": "P001", "name": "Widget A",   "category": "widgets",  "unit_price": 9.99},
    {"product_id": "P002", "name": "Widget B",   "category": "widgets",  "unit_price": 14.99},
    {"product_id": "P003", "name": "Gadget X",   "category": "gadgets",  "unit_price": 49.99},
    {"product_id": "P004", "name": "Gadget Y",   "category": "gadgets",  "unit_price": 79.99},
    {"product_id": "P005", "name": "Accessory Z","category": "accessories","unit_price": 4.99},
]
STATUSES = ["completed", "completed", "completed", "pending", "refunded"]  # weighted
CUSTOMERS = [f"CUST{i:04d}" for i in range(1, 21)]  # 20 synthetic customers

random.seed(42)  # reproducible

@dlt.resource(primary_key="order_id", write_disposition="replace")
def orders():
    """Yields 100 synthetic e-commerce orders."""
    start = date(2024, 1, 1)
    for i in range(1, 101):
        product = random.choice(PRODUCTS)
        qty = random.randint(1, 5)
        yield {
            "order_id":    f"ORD{i:05d}",
            "customer_id": random.choice(CUSTOMERS),
            "product_id":  product["product_id"],
            "product_name":product["name"],
            "category":    product["category"],
            "unit_price":  product["unit_price"],
            "quantity":    qty,
            "total_amount":round(product["unit_price"] * qty, 2),
            "status":      random.choice(STATUSES),
            "order_date":  str(start + timedelta(days=random.randint(0, 179))),
        }

# Create the pipeline — everything lands in DuckDB
pipeline = dlt.pipeline(
    pipeline_name="ecommerce",
    destination="duckdb",
    dataset_name="raw",
)

info = pipeline.run(orders())
print(info)

**What just happened?**
- 100 synthetic orders landed in `raw.orders` inside `ecommerce.duckdb`
- dlt inferred all column types (strings, floats, ints)
- `write_disposition="replace"` means each run rebuilds the table — no duplicates

Let's verify the raw data:

In [ ]:
# Quick sanity check on the raw data
with pipeline.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) as total, SUM(total_amount) as revenue FROM raw.orders") as cur:
        print("Raw order count and revenue:")
        print(cur.df())

    with client.execute_query("SELECT * FROM raw.orders LIMIT 3") as cur:
        print("\nSample rows:")
        print(cur.df().to_string())

---
## Step 3 · Post-load SQL transformations with `pipeline.sql_client()`

Once data is loaded, `pipeline.sql_client()` gives you a connection to run any SQL against the destination. This is how you build summary tables, clean data, or create analytics-ready views — all in Python, no separate BI tool needed.

In [ ]:
# Build a category summary table using post-load SQL
# This is a pure SQL transform — dlt just provides the connection

CREATE_SUMMARY = """
CREATE OR REPLACE TABLE raw.category_summary AS
SELECT
    category,
    COUNT(*)                                        AS total_orders,
    SUM(quantity)                                   AS total_units,
    ROUND(SUM(total_amount), 2)                     AS total_revenue,
    ROUND(AVG(total_amount), 2)                     AS avg_order_value,
    COUNT(DISTINCT customer_id)                     AS unique_customers,
    SUM(CASE WHEN status = 'refunded' THEN 1 END)   AS refunded_orders
FROM raw.orders
WHERE status != 'pending'
GROUP BY category
ORDER BY total_revenue DESC
"""

CREATE_CUSTOMER_SUMMARY = """
CREATE OR REPLACE TABLE raw.customer_lifetime_value AS
SELECT
    customer_id,
    COUNT(*)                            AS total_orders,
    ROUND(SUM(total_amount), 2)         AS lifetime_value,
    ROUND(AVG(total_amount), 2)         AS avg_order_value,
    MIN(order_date)                     AS first_order_date,
    MAX(order_date)                     AS last_order_date
FROM raw.orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY lifetime_value DESC
"""

# Execute both transforms using the same DuckDB connection dlt manages
with pipeline.sql_client() as client:
    client.execute_sql(CREATE_SUMMARY)
    print("category_summary table created.")

    client.execute_sql(CREATE_CUSTOMER_SUMMARY)
    print("customer_lifetime_value table created.")

    # Inspect the results
    with client.execute_query("SELECT * FROM raw.category_summary") as cur:
        print("\n── Category Summary ──")
        print(cur.df().to_string(index=False))

    with client.execute_query("SELECT * FROM raw.customer_lifetime_value LIMIT 5") as cur:
        print("\n── Top 5 Customers by LTV ──")
        print(cur.df().to_string(index=False))

**What just happened?**
- `pipeline.sql_client()` opens a connection to the *same* DuckDB file dlt wrote to
- `client.execute_sql()` runs DDL — `CREATE OR REPLACE TABLE` is idempotent (safe to re-run)
- `client.execute_query()` returns a cursor; `.df()` gives you a pandas DataFrame
- The transform tables live in the same `raw` dataset — in production you'd separate them into a `transformed` or `marts` dataset

This pattern works with **any dlt destination** — BigQuery, Snowflake, Redshift. The SQL changes; the Python pattern stays identical.

---
## Step 4 · In-flight transforms with `@dlt.transformer`

Sometimes you need to reshape records *before* they land — normalise a field, split a nested dict, or derive a computed column. Use `@dlt.transformer` to chain a transform onto a resource.

In [ ]:
import dlt

# Source resource: raw orders (same structure as before)
@dlt.resource(primary_key="order_id", write_disposition="replace")
def raw_orders():
    """Yields raw order records — some with messy data."""
    yield [
        {"order_id": "ORD001", "customer_id": "CUST01", "product_id": "P001",
         "unit_price": 9.99,  "quantity": 2, "status": "COMPLETED",  # status is UPPERCASE
         "order_date": "2024-03-15", "discount_pct": 0.10},
        {"order_id": "ORD002", "customer_id": "CUST02", "product_id": "P003",
         "unit_price": 49.99, "quantity": 1, "status": "pending",
         "order_date": "2024-03-16", "discount_pct": 0.0},
        {"order_id": "ORD003", "customer_id": "CUST01", "product_id": "P005",
         "unit_price": 4.99,  "quantity": 3, "status": "REFUNDED",   # UPPERCASE again
         "order_date": "2024-03-17", "discount_pct": 0.05},
    ]

# Transformer: runs on each record from raw_orders before it's written
@dlt.transformer(primary_key="order_id", write_disposition="replace")
def clean_orders(record):
    """
    Enriches and normalises each raw order record.
    Called once per record (or once per batch item if input is a list).
    """
    # Normalise status to lowercase
    record["status"] = record["status"].lower()

    # Derive computed columns
    gross = record["unit_price"] * record["quantity"]
    discount = gross * record.get("discount_pct", 0.0)
    record["gross_amount"]    = round(gross, 2)
    record["discount_amount"] = round(discount, 2)
    record["net_amount"]      = round(gross - discount, 2)

    # Remove the raw field we've already used
    record.pop("discount_pct", None)

    # Yield the cleaned record — transformers always yield
    yield record

# Wire them together: raw_orders feeds into clean_orders
# The pipe operator (|) chains resource → transformer
transform_pipeline = dlt.pipeline(
    pipeline_name="ecommerce_clean",
    destination="duckdb",
    dataset_name="clean",
)

info = transform_pipeline.run(raw_orders() | clean_orders)
print(info)

In [ ]:
# Inspect the transformed data — statuses are lowercase, computed columns are present
with transform_pipeline.sql_client() as client:
    with client.execute_query("SELECT order_id, status, gross_amount, discount_amount, net_amount FROM clean.clean_orders") as cur:
        print(cur.df().to_string(index=False))

**What just happened?**
- `@dlt.transformer` receives records one-by-one from the upstream resource
- The `|` (pipe) operator wires `raw_orders()` → `clean_orders` — this is dlt's functional composition pattern
- The transformer *yields* records, so it can also expand one record into many (e.g. splitting a batch)
- No raw data was written to the destination — only the cleaned output

**When to use in-flight vs post-load transforms:**

| Use in-flight (`@dlt.transformer`) when... | Use post-load SQL when... |
|---|---|
| You need to reshape or split records | You need aggregates across many rows |
| Logic is per-record, pure Python | Logic is SQL-native (window functions, JOINs) |
| You don't want raw data in the warehouse | Raw data is valuable to keep |

---
## Step 5 · dbt integration

dbt is the industry standard for SQL-based transformations with lineage, tests, and documentation. dlt integrates cleanly as the **loading** step — dbt takes over for **transformation**.

### How they connect

```
dlt pipeline.run()  →  DuckDB file  ←  dbt reads via profiles.yml
```

dlt writes to a `.duckdb` file. dbt's DuckDB adapter reads from the same file. They share the database — dlt owns the raw schema, dbt owns the transform schemas.

### Setting up a dbt project

> **Note:** dbt requires a separate install (`pip install dbt-duckdb`). The cells below show the setup commands and configuration. Run them in a terminal, not this notebook, unless you want to install dbt into the notebook environment.

```bash
# Install dbt with the DuckDB adapter
pip install dbt-duckdb

# Initialise a new dbt project
dbt init ecommerce_transforms
cd ecommerce_transforms
```

### `profiles.yml` — point dbt at dlt's DuckDB file

dbt uses `profiles.yml` (usually at `~/.dbt/profiles.yml`) to find the database. Point it at the `.duckdb` file dlt created:

```yaml
# ~/.dbt/profiles.yml
ecommerce_transforms:
  target: dev
  outputs:
    dev:
      type: duckdb
      # Path to the DuckDB file dlt created
      # dlt names it after the pipeline: {pipeline_name}.duckdb
      path: /path/to/your/ecommerce.duckdb
      schema: transformed   # dbt writes to this schema
      threads: 4
```

### Example dbt model

Create `models/marts/category_revenue.sql`:

```sql
-- models/marts/category_revenue.sql
-- ref() tells dbt about lineage — it knows this model depends on raw.orders
{{ config(materialized='table') }}

SELECT
    category,
    COUNT(*)                        AS total_orders,
    ROUND(SUM(total_amount), 2)     AS total_revenue,
    ROUND(AVG(total_amount), 2)     AS avg_order_value
FROM {{ source('raw', 'orders') }}
WHERE status = 'completed'
GROUP BY category
```

`sources.yml` registers the dlt-loaded tables:

```yaml
# models/sources.yml
version: 2
sources:
  - name: raw
    schema: raw
    tables:
      - name: orders
      - name: category_summary
```

---
## Step 6 · Run dbt programmatically from Python

You can trigger `dbt run` from Python after your dlt pipeline completes. This keeps the full ELT in one script.

In [ ]:
import subprocess
import sys
import os

def run_dbt_after_load(dbt_project_dir: str, target: str = "dev") -> bool:
    """
    Runs 'dbt run' after a successful dlt load.

    Call this function after pipeline.run() completes.
    Returns True on success, False on failure.

    Args:
        dbt_project_dir: Absolute path to the dbt project directory
        target: dbt target profile name (default: 'dev')
    """
    print(f"Running dbt in: {dbt_project_dir}")

    result = subprocess.run(
        [sys.executable, "-m", "dbt", "run", "--target", target],
        cwd=dbt_project_dir,
        capture_output=True,
        text=True,
    )

    if result.returncode == 0:
        print("dbt run succeeded.")
        print(result.stdout[-2000:])  # last 2000 chars to avoid overflow
        return True
    else:
        print("dbt run FAILED.")
        print(result.stderr[-2000:])
        return False


def full_elt_pipeline(dbt_project_dir: str = None):
    """
    Complete ELT pipeline:
    1. dlt: Extract + Load raw data
    2. dlt SQL client: Lightweight post-load transforms
    3. dbt (optional): Full transformation layer
    """
    # ── Step 1: Load raw data with dlt ──
    pipeline = dlt.pipeline(
        pipeline_name="ecommerce",
        destination="duckdb",
        dataset_name="raw",
    )
    info = pipeline.run(orders())
    print(f"Load complete: {info}")

    # ── Step 2: Post-load SQL transforms ──
    with pipeline.sql_client() as client:
        client.execute_sql(CREATE_SUMMARY)            # from Step 3 above
        client.execute_sql(CREATE_CUSTOMER_SUMMARY)   # from Step 3 above
    print("SQL transforms applied.")

    # ── Step 3: dbt transforms (if project path provided) ──
    if dbt_project_dir and os.path.isdir(dbt_project_dir):
        success = run_dbt_after_load(dbt_project_dir)
        if not success:
            raise RuntimeError("dbt run failed — check logs above")
    else:
        print("No dbt project path provided — skipping dbt step.")

    return info


# Run the full ELT pipeline (without dbt since we don't have a project here)
result = full_elt_pipeline()
print("\nELT pipeline complete!")

**What just happened?**
- `full_elt_pipeline()` orchestrates the complete ELT in 3 steps
- Step 1: dlt extracts and loads raw data
- Step 2: `sql_client()` applies lightweight SQL transforms
- Step 3: dbt runs if a project path is given — `subprocess.run()` calls dbt as a subprocess
- In production, you'd replace `dbt_project_dir=None` with the real path

### Complete dlt → dbt ELT architecture

```
┌─────────────────────────────────────────────────────────┐
│  Python script / Airflow DAG / GitHub Actions           │
│                                                         │
│  pipeline.run(orders())          ← dlt: Extract + Load  │
│       ↓                                                 │
│  sql_client().execute_sql(...)   ← dlt: Quick transforms│
│       ↓                                                 │
│  subprocess dbt run              ← dbt: Full transforms │
│       ↓                                                 │
│  DuckDB / BigQuery / Snowflake                          │
│    ├── raw.orders          (dlt writes)                 │
│    ├── raw.category_summary(dlt SQL client writes)      │
│    └── transformed.category_revenue (dbt writes)        │
└─────────────────────────────────────────────────────────┘
```

---
## Challenge — do this yourself

1. Add a new transformer called `tag_high_value` that takes records from `raw_orders()` and adds a boolean field `is_high_value` (True if `net_amount > 30`)
2. Chain it after `clean_orders`: `raw_orders() | clean_orders | tag_high_value`
3. Load the result and query how many high-value orders exist

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
@dlt.transformer(primary_key="order_id", write_disposition="replace")
def tag_high_value(record):
    record["is_high_value"] = record.get("net_amount", 0) > 30
    yield record

pipeline_hv = dlt.pipeline(
    pipeline_name="ecommerce_hv",
    destination="duckdb",
    dataset_name="enriched",
)

info = pipeline_hv.run(raw_orders() | clean_orders | tag_high_value)
print(info)

with pipeline_hv.sql_client() as client:
    with client.execute_query(
        "SELECT is_high_value, COUNT(*) as cnt FROM enriched.tag_high_value GROUP BY 1"
    ) as cur:
        print(cur.df())
```

Three transformers chained: raw → clean → tag. Each transformer is a pure function — easy to test individually.
</details>

---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| ELT pattern | Extract + Load first; transform inside the warehouse |
| `pipeline.sql_client()` | Opens a connection to the destination for post-load SQL |
| `client.execute_sql()` | DDL and DML — CREATE, INSERT, UPDATE, DROP |
| `client.execute_query()` | SELECT — returns a cursor with `.df()` for pandas |
| `@dlt.transformer` | In-flight per-record transform; use `|` to chain |
| `resource | transformer` | Pipe syntax — records flow from resource into transformer |
| dbt integration | `profiles.yml` → point `path:` at dlt's `.duckdb` file |
| dbt from Python | `subprocess.run(["dbt", "run"])` after `pipeline.run()` |

> **Tip:** Keep raw data raw. Use dbt or SQL client transforms to build analytics-ready tables. If you need to fix data *before* it lands, use `@dlt.transformer`.

---
## What's next → Day 7

**Day 7** → Error handling and data quality: schema contracts, Pydantic validation, filtering bad records, and reading `pipeline.last_trace`.

Mark Day 6 complete in your [tracker](../index.html).